# Mean-Expected Shortfall Portfolios

The Expected Shortfall at tail probability $\alpha$ is the average loss on the worst $\alpha$ share of days, a coherent risk measure (Acerbi and Tasche 2002). Rockafellar and Uryasev (2000) showed that minimising it over the weights is a linear programme when the observed days are used as scenarios. This notebook compares the minimum Expected Shortfall portfolio with the minimum-variance one.

In [ ]:
Pollis.packages("PortfolioOptimisers", "Clarabel", "Plots")

In [ ]:
using PortfolioOptimisers, Clarabel, Random, Statistics, Plots

## The data

1,000 trading days of daily simple returns for 8 assets in two sectors. A market factor moves every asset, a sector factor moves each sector, and Student-t shocks with 5 degrees of freedom give fat tails. The true annualised means are 12% for Tech and 6% for Util, so the estimates can be checked against them.

In [ ]:
Random.seed!(42)
T, N = 1_000, 8
t5(n) = sqrt(3 / 5) .* randn(n) ./ sqrt.(vec(sum(abs2, randn(5, n); dims = 1)) ./ 5)   # unit-variance Student-t(5) shocks
mkt = 0.008 .* t5(T); tech = 0.010 .* t5(T); util = 0.005 .* t5(T)   # market and sector factors
mu_d = [fill(0.12, 4); fill(0.06, 4)] ./ 252   # true daily means
R = hcat([j <= 4 ? mu_d[j] .+ 1.2 .* mkt .+ tech .+ 0.012 .* t5(T) : mu_d[j] .+ 0.6 .* mkt .+ util .+ 0.006 .* t5(T) for j in 1:N]...)
asset_names = ["Tech1", "Tech2", "Tech3", "Tech4", "Util1", "Util2", "Util3", "Util4"]

rd = ReturnsResult(; nx = asset_names, X = R)
rf_daily = 0.02 / 252
slv = Solver(; name = :clarabel, solver = Clarabel.Optimizer, settings = Dict("verbose" => false),
             check_sol = (; allow_local = true, allow_almost = true))
show_weights(w) = foreach((nm, wi) -> println(rpad(nm, 8), lpad(round(100 * wi; digits = 2), 8), " %"), asset_names, w)

## Minimum Expected Shortfall

With $\alpha = 0.05$, the measure averages the worst 50 of the 1,000 days.

In [ ]:
es5 = ConditionalValueatRisk(; alpha = 0.05)
res = optimise(MeanRisk(; opt = JuMPOptimiser(; slv = slv), r = es5, obj = MinimumRisk()), rd)
w_es, pr = res.w, res.pr
show_weights(w_es)

## Against minimum variance

Each portfolio is best by its own measure. The table shows how much each loses by the other's.

In [ ]:
w_minv = optimise(MeanRisk(; opt = JuMPOptimiser(; slv = slv), r = Variance(), obj = MinimumRisk()), rd).w
println(rpad("", 18), lpad("Vol %", 8), lpad("ES 5% %", 10))
for (nm, w) in [("Minimum ES", w_es), ("Minimum Variance", w_minv)]
    println(rpad(nm, 18), lpad(round(100 * sqrt(252) * expected_risk(StandardDeviation(), w, pr); digits = 2), 8),
            lpad(round(100 * expected_risk(es5, w, pr); digits = 3), 10))
end
bar(asset_names, 100 .* w_es; label = "Minimum ES", ylabel = "Weight (%)", xrotation = 45)
scatter!(asset_names, 100 .* w_minv; label = "Minimum Variance", ms = 6)

## The worst days

A histogram of the minimum-ES portfolio's daily returns, with the threshold of the worst 5% marked.

In [ ]:
rp = R * w_es
q = quantile(rp, 0.05)
histogram(100 .* rp; bins = 60, label = "Daily Returns", xlabel = "Return (%)")
vline!([100 * q]; label = "Worst 5% Threshold", ls = :dash)
println("Average return on the worst 5% of days: ", round(100 * mean(rp[rp .<= q]); digits = 3), " %")

## Exercises

1. Set `alpha = 0.01`. How many days does the measure now rest on, and how much do the weights change?
2. Replace `MinimumRisk()` with `MaximumRatio(; rf = rf_daily)`, the STARR ratio. How do the weights compare with the maximum Sharpe portfolio?
3. Change the shocks to normal ones, redefining `t5(n) = randn(n)`. Do the minimum-ES and minimum-variance weights move closer?